# Spatial weights on Ohio counties (demo)

The data file is `ohio_counties_acs.gpkg`. Its columns are described in `ohio_counties_acs_dictionary.md` in the same data folder. Read that page before Thursday.


## Setup as usual

In [ ]:
# Run this cell first (takes ~1 minute). Colab already includes geopandas.
%pip install -q libpysal esda splot mapclassify contextily

## 1. Read the file and map the variable

Look for: 88 rows, a geographic CRS, and the column names from the data dictionary. The map is a quantile choropleth of median household income.

In [ ]:
DATA_BASE_P2 = "https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/data/part2"

# Local path fallback (uncomment if running outside Colab without internet access):
# DATA_BASE_P2 = "../data/part2"
# DATA_BASE_P2 = "/content"   # Colab: after uploading the file through the Files panel

import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

ohio = gpd.read_file(f"{DATA_BASE_P2}/ohio_counties_acs.gpkg")
print(ohio.shape, ohio.crs)
print(list(ohio.columns))

fig, ax = plt.subplots(figsize=(7, 7))
ohio.plot(column="median_hh_income", scheme="quantiles", k=5, cmap="YlGnBu",
          legend=True, edgecolor="white", linewidth=0.4, ax=ax,
          legend_kwds={"title": "Median household income (USD)", "loc": "lower left"})
ax.set_title("Median household income, Ohio counties (ACS 2019-2023)")
ax.set_axis_off()
plt.show()

## 2. Project

Contiguity does not need a projected CRS, but distance-based weights and anything that measures length do, so project once now. 

UTM zone 17N covers Ohio.

In [ ]:
ohio_p = ohio.to_crs(26917)
print(ohio_p.crs)
print(ohio_p.total_bounds.round(0))

## 3. Queen contiguity weights

Look for: `islands: []`. An island is a polygon with no neighbors, and it would break row-standardization. One connected component means every county can "reach" every other through neighbor links.

In [ ]:
from libpysal.weights import Queen

w = Queen.from_dataframe(ohio_p, use_index=False)
print("n:", w.n)
print("islands:", w.islands)
print("connected components:", w.n_components)

## 4. How many neighbors does each county have?

Look for: the distribution. 

Queen contiguity results in a variable number of neighbors; a county in the interior is more likely to have neighbors that one on the state line or the lake.

In [ ]:
cards = w.cardinalities                      # {row index: number of neighbors}
values = list(cards.values())
print("neighbors per county: min", min(values), "| max", max(values), "| mean", round(sum(values) / len(values), 2))

most = max(cards, key=cards.get)
fewest = min(cards, key=cards.get)
print(f"most neighbors:   {ohio_p.loc[most, 'name']} ({cards[most]})")
print(f"fewest neighbors: {ohio_p.loc[fewest, 'name']} ({cards[fewest]})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(values, bins=range(1, max(values) + 2), align="left", edgecolor="white", color="#4C72B0")
ax.set_xlabel("Number of Queen neighbors")
ax.set_ylabel("Number of counties")
ax.set_title("Queen contiguity cardinality, 88 Ohio counties")
plt.show()

## 5. Row-standardize

After row-standardization they are all 1 divided by the number of neighbors, and the row sums to 1. That is what turns a weighted sum into a weighted average.

In [ ]:
portage = ohio_p[ohio_p["name"] == "Portage"].index[0]
neighbor_names = [ohio_p.loc[j, "name"] for j in w.neighbors[portage]]
print("Portage's Queen neighbors:", neighbor_names)
print("raw weights:             ", w.weights[portage])

w.transform = "r"                           # row-standardize: each row now sums to 1
print("row-standardized weights:", [round(x, 3) for x in w.weights[portage]])

## 6. Spatial lag

The spatial lag of a variable is, for each county, the weighted average of that variable across its neighbors. 

Look for: Portage's own income compared to the average of its neighbors

In [ ]:
from libpysal.weights.spatial_lag import lag_spatial

ohio_p["income_lag"] = lag_spatial(w, ohio_p["median_hh_income"].values)

own = ohio_p.loc[portage, "median_hh_income"]
lag = ohio_p.loc[portage, "income_lag"]
print(f"Portage County: median household income ${own:,.0f}; its neighbors average ${lag:,.0f}.")
ohio_p[["name", "median_hh_income", "income_lag"]].sort_values("median_hh_income", ascending=False).head(8)

## 7. Each county against its neighbors

Look for: the slop of the fitted line. The dashed lines mark the means and divide the plot into the four quadrants from the slides.

In [ ]:
x = ohio_p["median_hh_income"].values
y = ohio_p["income_lag"].values
slope, intercept = np.polyfit(x, y, 1)
corr = np.corrcoef(x, y)[0, 1]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x, y, s=25, color="#4C72B0", alpha=0.8)
xs = np.array([x.min(), x.max()])
ax.plot(xs, slope * xs + intercept, color="firebrick")
ax.axvline(x.mean(), color="gray", linestyle="--", linewidth=0.8)
ax.axhline(y.mean(), color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Median household income (USD)")
ax.set_ylabel("Neighbors' average median household income (USD)")
ax.set_title("Income vs. neighbors' average income, Ohio counties")
plt.show()

print("slope of the fitted line (lag on value):", round(slope, 4))
print("correlation between income and its spatial lag:", round(corr, 3))
print("spread of income:", round(x.std()), "| spread of the lag:", round(y.std()), "| ratio:", round(y.std() / x.std(), 3))

The slope of that line is Moran's I. 

The correlation is a different number (and it is larger). 

Correlation divides by the spread of both axes; the slope divides only by the spread of the values on the x-axis. 

A spatial lag is an average over neighbors, so it varies less than the raw values (the ratio printed above), and the slope is the correlation shrunk by that ratio.

## 8. As a single function call

Everything above, from the weights to the slope, is what `esda.Moran` does internally. 

One function call returns I, along with a permutation test of whether a value that large could arise by chance.

 Look for: an I close to the correlation printed above (not identical, since I uses standardized values and a slightly different scaling), and a `p_sim` of 0.001, the smallest value 999 permutations can produce.

In [ ]:
from esda.moran import Moran

np.random.seed(42)                          # set the seed immediately before the permutation test
mi = Moran(ohio_p["median_hh_income"].values, w, permutations=999)

print("Moran's I:", round(mi.I, 4))
print("expected I under spatial randomness:", round(mi.EI, 4))
print("p-value (999 permutations):", mi.p_sim)